1. Composition is the mapping
$$
  i \mapsto n_i
$$
or
$$
  i \mapsto m_i
$$
2.  
$$
  x_i = \frac{n_i}{\sum_j n_j}
$$
$$
  w_i = \frac{m_i}{\sum_j m_j}
$$


In [ ]:
from __future__ import annotations
from dataclasses import dataclass
from typing import Dict

import numpy as np

@dataclass(frozen=True)
class MolarMixture:
    """
    Composition defined by mole amounts n_i.

    species -> moles
    """
    n: Dict[str, float]  # mol
    molar_masses: Dict[str, float]  # kg/mol

    # -------------------------
    # basic properties
    # -------------------------
    @property
    def n_tot(self) -> float:
        return float(sum(self.n.values()))

    @property
    def x(self) -> Dict[str, float]:
        nt = self.n_tot
        return {k: v/nt for k, v in self.n.items()}

    # -------------------------
    # conversions
    # -------------------------
    def to_mass_mixture(self) -> "MassMixture":
        m = {
            k: self.n[k]*self.molar_masses[k]
            for k in self.n
        }
        return MassMixture(m=m, molar_masses=self.molar_masses)

    # -------------------------
    # utilities
    # -------------------------
    def mean_molar_mass(self) -> float:
        """
        Mixture-average molar mass:

        M̄ = Σ x_i M_i
        """
        x = self.x
        return sum(x[k]*self.molar_masses[k] for k in x)

@dataclass(frozen=True)
class MassMixture:
    """
    Composition defined by masses m_i.

    species -> kg
    """
    m: Dict[str, float]  # kg
    molar_masses: Dict[str, float]  # kg/mol

    @property
    def m_tot(self) -> float:
        return float(sum(self.m.values()))

    @property
    def w(self) -> Dict[str, float]:
        mt = self.m_tot
        return {k: v/mt for k, v in self.m.items()}

    def to_molar_mixture(self) -> MolarMixture:
        n = {
            k: self.m[k]/self.molar_masses[k]
            for k in self.m
        }
        return MolarMixture(n=n, molar_masses=self.molar_masses)

    def mean_molar_mass(self) -> float:
        """
        M̄ = 1 / Σ (w_i / M_i)
        """
        w = self.w
        return 1.0 / sum(
            w[k]/self.molar_masses[k] for k in w
        )
